# **Start**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/noise_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d06_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # noise train
    shutil.copy(glob.glob("/kaggle/input/**/d06_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # noise test
    print("Kaggle: noise data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: noise data copied to ./drive/MyDrive/Regression/noise_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks (it has numpy/sklearn/torch/xgboost/
# lightgbm/catboost/optuna/shap already), and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 bayesian-optimization==3.2.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
import gpboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold  # FIX 1
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from pytorch_tabnet import TabNetRegressor

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


In [4]:
# Go to find & replace button and replace (noise_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [5]:
feature_names = ['A', 'B', 'C', 'D', 'O']
TARGET_LABEL = 'Noise level (Leq)'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [6]:
train_data_path = "./drive/MyDrive/Regression/noise_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/noise_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (168, 6)
First 5 rows of training data:
       A    B    C    D   O    Leq
0   894  354  228   30  42  70.06
1  1068  312  234   24  24  69.14
2  1506  894  198  222  18  73.02
3  1188  498  252  354  18  75.71
4  1068  438  210  366  30  70.78

Shape of test data: (60, 6)
First 5 rows of test data:
       A    B    C    D   O    Leq
0  1032  282   90  108   6  70.07
1  1188  498  252  408  18  72.90
2   570   90   60   84  24  68.66
3   450  156  150   72   0  50.24
4   240   66   18   24   0  40.17


In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (168, 5)
Shape of y_train: (168,)
Shape of X_test: (60, 5)
Shape of y_test: (60,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[-0.34508703 -0.52871218  0.0597039  -1.05667951  1.31930842]
 [-0.08377773 -0.63289541  0.09903824 -1.09452265  0.25541811]
 [ 0.57400083  0.8107865  -0.13696778  0.15430089 -0.09921199]
 [ 0.09643557 -0.17151253  0.21704125  0.98684991 -0.09921199]
 [-0.08377773 -0.32034572 -0.05829911  1.06253618  0.61004822]]

First five rows of normalized X_test:
[[-0.13784172 -0.707312   -0.84498584 -0.56471872 -0.8084722 ]
 [ 0.09643557 -0.17151253  0.21704125  1.32743815 -0.09921199]
 [-0.83166295 -1.1835782  -1.04165752 -0.71609127  0.25541811]
 [-1.01187626 -1.01986169 -0.45164247 -0.79177755 -1.16310231]
 [-1.32724954 -1.24311147 -1.31699788 -1.09452265 -1.16310231]]


# **Functions**

In [10]:
# FIX B1: helper was used in this notebook but never defined (NameError after the first model).
# Copied unchanged from the other notebooks of this project.
def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def plot_best_scores(best_scores_ran, excel_file_path):
    # Extract the best pruner for each model based on RMSE and correlation coefficient
    best_rmse_scores = {}
    best_corr_coef_scores = {}

    for (model_name, pruner_name), scores in best_scores_ran.items():
        # Initialize if not already present
        if model_name not in best_rmse_scores:
            best_rmse_scores[model_name] = (scores['test_rmse'], pruner_name)
        if model_name not in best_corr_coef_scores:
            best_corr_coef_scores[model_name] = (scores['test_corr_coef'], pruner_name)

        # Update if better scores are found
        if scores['test_rmse'] < best_rmse_scores[model_name][0]:
            best_rmse_scores[model_name] = (scores['test_rmse'], pruner_name)
        if scores['test_corr_coef'] > best_corr_coef_scores[model_name][0]:
            best_corr_coef_scores[model_name] = (scores['test_corr_coef'], pruner_name)

    # Prepare data for plotting
    model_names_rmse = [f"{model} ({pruner})" for model, (rmse, pruner) in best_rmse_scores.items()]
    rmse_values = [rmse for rmse, _ in best_rmse_scores.values()]

    model_names_corr = [f"{model} ({pruner})" for model, (corr, pruner) in best_corr_coef_scores.items()]
    corr_values = [corr for corr, _ in best_corr_coef_scores.values()]

    # Plot RMSE
    plt.figure(figsize=(12, 6))
    bars_rmse = plt.bar(model_names_rmse, rmse_values, color='skyblue')

    # Highlight the best model
    best_rmse_index = np.argmin(rmse_values)
    bars_rmse[best_rmse_index].set_color('orange')

    # Annotate the bars with the RMSE scores
    for i, bar in enumerate(bars_rmse):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 0.05,
                 f'{rmse_values[i]:.5f}', ha='center', va='bottom', color='black')

    # Add labels and title for the RMSE bar chart
    plt.xlabel('Model (Pruner)')
    plt.ylabel('Test RMSE')
    plt.title('Best Test RMSE for Each Model')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    rmse_image_path = 'rmse_plot.png'
    _ensure_parent_dir(rmse_image_path)
    plt.savefig(_ensure_parent_dir(rmse_image_path))
    plt.close()

    # Plot Correlation Coefficient
    plt.figure(figsize=(12, 6))
    bars_corr = plt.bar(model_names_corr, corr_values, color='lightgreen')

    # Highlight the best model
    best_corr_index = np.argmax(corr_values)
    bars_corr[best_corr_index].set_color('orange')

    # Annotate the bars with the correlation coefficient scores
    for i, bar in enumerate(bars_corr):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 0.05,
                 f'{corr_values[i]:.5f}', ha='center', va='bottom', color='black')

    # Add labels and title for the correlation coefficient bar chart
    plt.xlabel('Model (Pruner)')
    plt.ylabel('Correlation Coefficient')
    plt.title('Best Correlation Coefficient for Each Model')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    corr_image_path = 'corr_plot.png'
    _ensure_parent_dir(corr_image_path)
    plt.savefig(_ensure_parent_dir(corr_image_path))
    plt.close()

    # Load the existing Excel file
    workbook = load_workbook(_ensure_excel_file(excel_file_path))

    # Create a new sheet for the plots
    sheet_name = 'Best Models Plots'
    if sheet_name in workbook.sheetnames:
        sheet = workbook[sheet_name]
    else:
        sheet = workbook.create_sheet(title=sheet_name)

    # Insert images into the new Excel sheet
    img_rmse = Image(rmse_image_path)
    img_corr = Image(corr_image_path)

    # Insert images
    sheet.add_image(img_rmse, 'A1')
    sheet.add_image(img_corr, 'A20')  # Adjust the position as needed

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    workbook.save(_ensure_parent_dir(excel_file_path))

    # Clean up the image files
    if os.path.exists(str(rmse_image_path)): os.remove(str(rmse_image_path))
    if os.path.exists(str(corr_image_path)): os.remove(str(corr_image_path))

# Example usage
# plot_best_scores(best_scores_ran, 'path_to_your_excel_file.xlsx')

In [11]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_interpretml_explanations_summary_pruners(
    results_dict, X_train, y_train, X_test, feature_names, instance_indices=None, excel_file_path=None
):
    if instance_indices is None:
        instance_indices = range(len(X_test))
    elif isinstance(instance_indices, int):
        instance_indices = [instance_indices]

    valid_indices = [idx for idx in instance_indices if 0 <= idx < len(X_test)]
    if not valid_indices:
        print("No valid instance indices provided.")
        return

    if isinstance(X_test, pd.DataFrame):
        instances_to_explain = X_test.iloc[valid_indices]
    else:
        instances_to_explain = X_test[valid_indices]

    best_model_pruners = {}
    for model_key, model_info in results_dict.items():
        if isinstance(model_key, tuple):
            model_name, pruner_name = model_key
        else:
            model_name = model_key
            pruner_name = None

        best_score = model_info.get('best_score')
        if best_score is None:
            print(f"No 'best_score' found for {model_key}. Skipping this combination.")
            continue

        if model_name not in best_model_pruners:
            best_model_pruners[model_name] = {
                'pruner_name': pruner_name,
                'model_info': model_info,
                'best_score': best_score
            }
        else:
            current_best_score = best_model_pruners[model_name]['best_score']
            if best_score < current_best_score:
                best_model_pruners[model_name] = {
                    'pruner_name': pruner_name,
                    'model_info': model_info,
                    'best_score': best_score
                }

    for model_name, info in best_model_pruners.items():
        pruner_name = info['pruner_name']
        model_info = info['model_info']
        best_params = dict(model_info['best_params'])  # don't mutate original!
        model_class = model_classes.get(model_name)

        if model_class is None:
            print(f"Model {model_name} is not supported or not available.")
            continue

        if model_name == 'CatBoost':
            best_params['verbose'] = 0

        # ------- Main model fit logic ---------
        if model_name == "TabNet":
            # TabNet: reshape y, fit, flatten pred for LIME/SHAP, etc.
            y_train_tabnet = np.array(y_train).reshape(-1, 1)
            try:
                model = model_class(**{k: v for k, v in best_params.items() if k != "verbose"})
            except TypeError:
                model = model_class()
            model.fit(np.array(X_train), y_train_tabnet, max_epochs=100, patience=10, batch_size=1024, eval_set=[(np.array(X_train), y_train_tabnet)])
            def predict_fn(data):
                preds = model.predict(np.array(data))
                # flatten for interpreters
                return preds.flatten()
        else:
            try:
                model = model_class(**best_params)
            except TypeError:
                model = model_class()
            model.fit(X_train, y_train)
            def predict_fn(data):
                return model.predict(data)

        if isinstance(X_train, pd.DataFrame):
            data_for_explainer = X_train.values
        else:
            data_for_explainer = X_train

        if isinstance(instances_to_explain, pd.DataFrame):
            data_for_explanation = instances_to_explain.values
        else:
            data_for_explanation = instances_to_explain

        # Generate LIME explanations
        lime_explainer = LimeTabular(
            predict_fn,
            data=data_for_explainer,
            feature_names=feature_names,
            random_state=1,
            mode='regression'
        )
        lime_explanation = lime_explainer.explain_local(data_for_explanation)

        feature_importances_lime = {}
        num_instances = len(valid_indices)
        for idx in range(num_instances):
            explanation = lime_explanation.data(idx)
            for feature_name, feature_score in zip(explanation['names'], explanation['scores']):
                feature_importances_lime[feature_name] = feature_importances_lime.get(feature_name, 0) + abs(feature_score)
        feature_importances_lime = {k: v / num_instances for k, v in feature_importances_lime.items()}
        feature_importances_lime = {k: round(v, 3) for k, v in feature_importances_lime.items()}

        # Generate SHAP explanations using ShapKernel
        try:
            shap_explainer = ShapKernel(predict_fn, data_for_explainer, feature_names=feature_names)
            shap_explanation = shap_explainer.explain_local(data_for_explanation)

            feature_importances_shap = {}
            for idx in range(num_instances):
                explanation = shap_explanation.data(idx)
                for feature_name, feature_score in zip(explanation['names'], explanation['scores']):
                    feature_importances_shap[feature_name] = feature_importances_shap.get(feature_name, 0) + abs(feature_score)

            feature_importances_shap = {k: v / num_instances for k, v in feature_importances_shap.items()}
            feature_importances_shap = {k: round(v, 3) for k, v in feature_importances_shap.items()}
        except Exception as e:
            print(f"Could not compute SHAP values for model {model_name}: {e}")
            feature_importances_shap = {}

        # Plot LIME and SHAP feature importances side by side
        fig, axes = plt.subplots(1, 2, figsize=(34, 36))

        # Plot LIME feature importances
        lime_importances_df = pd.DataFrame.from_dict(
            feature_importances_lime, orient='index', columns=['importance']
        )
        lime_importances_df.sort_values(by='importance', ascending=False, inplace=True)
        lime_importances_df.plot(kind='bar', legend=False, color='skyblue', ax=axes[0])
        axes[0].set_title(f"LIME Feature Importances for {model_name}")
        axes[0].set_ylabel("Average Absolute Importance Score")
        axes[0].set_xlabel("Features")
        axes[0].tick_params(axis='x', rotation=45)

        for p in axes[0].patches:
            height = p.get_height()
            axes[0].annotate(f'{height:.3f}',
                             (p.get_x() + p.get_width() / 2., height),
                             ha='center', va='bottom', fontsize=8)

        # Plot SHAP feature importances
        shap_importances_df = pd.DataFrame.from_dict(
            feature_importances_shap, orient='index', columns=['importance']
        )
        shap_importances_df.sort_values(by='importance', ascending=False, inplace=True)
        shap_importances_df.plot(kind='bar', legend=False, color='orange', ax=axes[1])
        axes[1].set_title(f"SHAP Feature Importances for {model_name}")
        axes[1].set_ylabel("Average Absolute SHAP Value")
        axes[1].set_xlabel("Features")
        axes[1].tick_params(axis='x', rotation=45)

        for p in axes[1].patches:
            height = p.get_height()
            axes[1].annotate(f'{height:.3f}',
                             (p.get_x() + p.get_width() / 2., height),
                             ha='center', va='bottom', fontsize=8)

        plt.tight_layout()

        # Save plots as images
        image_path = f'feature_importances_{model_name}.png'
        fig.savefig(_ensure_parent_dir(image_path))
        plt.close(fig)

        # Optionally insert images and scores into an Excel file
        if excel_file_path:
            workbook = load_workbook(_ensure_excel_file(excel_file_path))
            sheet_name = f'{model_name} Explanations'
            if sheet_name in workbook.sheetnames:
                sheet = workbook[sheet_name]
            else:
                sheet = workbook.create_sheet(title=sheet_name)

            # Insert images into the new Excel sheet
            img = Image(image_path)
            sheet.add_image(img, 'A1')

            # Create a new sheet for feature importance scores
            scores_sheet_name = f'{model_name} Scores'
            if scores_sheet_name in workbook.sheetnames:
                scores_sheet = workbook[scores_sheet_name]
            else:
                scores_sheet = workbook.create_sheet(title=scores_sheet_name)

            # Write LIME scores
            scores_sheet.append(['Feature', 'LIME Importance'])
            for feature, importance in feature_importances_lime.items():
                scores_sheet.append([feature, importance])

            # Write SHAP scores if available
            if feature_importances_shap:
                scores_sheet.append(['Feature', 'SHAP Importance'])
                for feature, importance in feature_importances_shap.items():
                    scores_sheet.append([feature, importance])

            # Save the workbook
            _ensure_parent_dir(excel_file_path)
            workbook.save(_ensure_parent_dir(excel_file_path))

            # Clean up the image file
            if os.path.exists(str(image_path)): os.remove(str(image_path))

# **Hyperparameter tuning using Autosampler by Optuna**

In [12]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
import joblib

def mseloss_objective(yhat, y, sample_weight=None):
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian


def rmseloss_metric(yhat, y, sample_weight=None):
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss


def hyperparameter_tuning_all(X_train, y_train, X_test, y_test, excel_path):

    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    y_test = np.array(y_test)

    # ================= NEW =================
    model_save_dir = "./drive/MyDrive/Regression/noise_uncertainty_analysis/hyperparameter_tuning/models"
    os.makedirs(model_save_dir, exist_ok=True)
    best_rmse_tracker = {}
    # =======================================

    models = {
        'Random Forest': (RandomForestRegressor, {
            'n_estimators': [100, 200, 300, 500, 700],
            'criterion': ['squared_error', 'absolute_error', 'friedman_mse', 'poisson'],
            'max_depth': [None, 10, 20, 30, 40],
            'min_samples_split': [2, 5, 10, 0.01],
            'min_samples_leaf': [1, 3, 5, 0.01],
            'min_weight_fraction_leaf': [0.0, 0.01, 0.1, 0.2],
            'max_features': [1.0, 'sqrt', 'log2', 0.3, 0.5],
            'max_leaf_nodes': [None, 50, 100, 200],
            'min_impurity_decrease': [0.0, 0.01, 0.1, 0.2],
            'n_jobs': [-1],
            'random_state': [42],
            'verbose': [0],
            'warm_start': [False],
            'ccp_alpha': [0.0, 0.001, 0.01, 0.05, 0.1]
        }),
        'Gradient Boosting': (GradientBoostingRegressor, {
            'loss': ['squared_error', 'absolute_error', 'huber'],  # FIX 3: 'quantile' removed (point model must target the mean)
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'n_estimators': [100, 200, 300, 500, 700],
            'subsample': [1.0, 0.9, 0.7, 0.5],
            'criterion': ['friedman_mse', 'squared_error'],
            'min_samples_split': [2, 5, 10, 0.01],
            'min_samples_leaf': [1, 3, 5, 0.01],
            'min_weight_fraction_leaf': [0.0, 0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7, 10],
            'min_impurity_decrease': [0.0, 0.01, 0.1],
            'init': [None],
            'random_state': [42],
            'max_features': [None, 'sqrt', 'log2', 0.5],
            'alpha': [0.9, 0.5, 0.1],
            'verbose': [0],
            'max_leaf_nodes': [None, 10, 30, 50],
            'warm_start': [False],
            'validation_fraction': [0.1],
            'n_iter_no_change': [None, 10, 20],
            'tol': [1e-4, 1e-3],
            'ccp_alpha': [0.0, 0.001, 0.01]
        }),
        'XGBoost': (XGBRegressor, {
            'n_estimators': [100, 200, 300, 400, 500],
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_depth': [3, 5, 7],
            'min_child_weight': [1, 3, 5],
            'gamma': [0, 0.1, 0.5, 1],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9],
            'colsample_bytree': [0.5, 0.7, 0.9],
            'colsample_bylevel': [0.5, 0.7, 0.9],
            'reg_alpha': [0, 0.01, 0.1, 1],
            'reg_lambda': [0.1, 1, 5, 10],
            'objective': ['reg:squarederror'],
            'random_state': [42],
            'n_jobs': [-1]
        }),
        'LightGBM': (LGBMRegressor, {
            'n_estimators': [100, 200, 300, 400, 500],
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'num_leaves': [15, 31, 63],
            'max_depth': [3, 5, 7, -1],
            'min_child_samples': [1, 5, 10, 20],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.5, 0.7, 0.9, 1.0],
            'reg_alpha': [0, 0.01, 0.1, 1],
            'reg_lambda': [0, 0.1, 1, 10],
            'min_child_weight': [1e-5, 1e-3, 1e-2, 1e-1],
            'bagging_freq': [0, 1, 5],
            'objective': ['regression'],
            'random_state': [42],
            'n_jobs': [-1],
            'verbose': [-1]
        }),
        'GPBoost': (GPBoostRegressor, {
            'n_estimators': [100, 200, 300, 400, 500],
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_depth': [3, 5, 7, -1],
            'num_leaves': [15, 31, 63],
            'min_child_samples': [1, 5, 10, 20],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.5, 0.7, 0.9, 1.0],
            'reg_alpha': [0, 0.1, 0.5, 1.0],
            'reg_lambda': [0, 0.1, 0.5, 1.0],
            'min_child_weight': [1e-5, 1e-3, 1e-2, 1e-1],
            'random_state': [42],
            'n_jobs': [-1],
            'verbose': [-1]
        }),
        'CatBoost': (CatBoostRegressor, {
            'iterations': [200, 500, 1000],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'depth': [4, 6, 8, 10],
            'l2_leaf_reg': [1, 3, 5, 7, 9],
            'border_count': [32, 64, 128],
            'min_data_in_leaf': [1, 5, 10, 20],
            'rsm': [0.6, 0.8, 1.0],
            'bagging_temperature': [0, 1, 10],
            'random_seed': [42],
            'verbose': [0]
        }),
        'NGBoost': (NGBRegressor, {
            'n_estimators': [200, 500, 1000],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'natural_gradient': [True, False],
            'minibatch_frac': [0.5, 0.7, 0.9, 1.0],
            'col_sample': [0.5, 0.7, 0.9, 1.0],
            'Dist': [Normal],
            'Score': [LogScore],
            'random_state': [42],
            'verbose': [0]
        }),
        'TabNet': (TabNetRegressor, {
            'n_d': [8, 16, 32, 64],
            'n_a': [8, 16, 32, 64],
            'n_steps': [3, 5, 7, 10],
            'gamma': [1.0, 1.3, 1.5, 2.0],
            'lambda_sparse': [1e-4, 1e-3, 1e-2],
            'optimizer_params': [{'lr': 2e-2}], # Fixed learning rate as recommended
            'mask_type': ['sparsemax', 'entmax'],
            'n_shared': [1, 2, 3],
            'n_independent': [1, 2, 3],
            'scheduler_params': [{"step_size": 10, "gamma": 0.9}],
            'scheduler_fn': [torch.optim.lr_scheduler.StepLR],
            'seed': [42],
            'verbose': [0]
        }),
        'HistGradientBoosting': (HistGradientBoostingRegressor, {
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_iter': [100, 200, 300, 400, 500],
            'max_depth': [3, 5, 7, None],
            'min_samples_leaf': [5, 10, 20],
            'max_leaf_nodes': [15, 31, 63, None],
            'l2_regularization': [0.0, 0.1, 0.5, 1.0],
            'max_bins': [64, 128, 255],
            'early_stopping': [True, False],
            'validation_fraction': [0.1, 0.2],
            'n_iter_no_change': [5, 10, 15],
            'loss': ['squared_error'],
            'random_state': [42],
            'verbose': [0]
        }),
        'PGBM': (PGBM, {})
    }

    pruners = [
        optuna.pruners.MedianPruner(),
        optuna.pruners.NopPruner(),
        optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=3),
        optuna.pruners.PercentilePruner(25.0),
        optuna.pruners.SuccessiveHalvingPruner(),
        optuna.pruners.HyperbandPruner(),
        optuna.pruners.ThresholdPruner(upper=float(np.var(y_train))),  # FIX 2: was lower=0.1 (pruned the BEST trials); now prunes trials worse than predicting the mean
        optuna.pruners.WilcoxonPruner()
    ]

    best_scores = {}
    predictions_df = pd.DataFrame({'Actual': y_test})
    timing_records = []
    pruning_records = []  # FIX 2: completed / pruned trial counts per model x pruner (new Excel sheet)

    # FIX 1: hyper-parameters are scored by 5-fold CV on the TRAINING data only.
    # The test set is used once, after tuning, to evaluate the chosen model.
    cv_splits = list(KFold(n_splits=5, shuffle=True, random_state=42).split(X_train))

    def _fit_model(model_name, model_class, params, X_fit, y_fit):
        if model_name == 'PGBM':
            m = PGBM()
            m.train((X_fit, y_fit), objective=mseloss_objective, metric=rmseloss_metric, params=params)
        else:
            m = model_class(**params)
            if model_name == 'TabNet':
                m.fit(X_fit, y_fit.reshape(-1, 1))
            else:
                m.fit(X_fit, y_fit)
        return m

    def _predict(model_name, m, X):
        p = m.predict(X)
        if torch.is_tensor(p):
            p = p.detach().cpu().numpy()
        return np.asarray(p).ravel()

    # =========================================================
    # FIX B14 (speed only, approved 2026-11-22). The search, the 5-fold CV, the pruners and the choice of the
    # best model (lowest CV error) are unchanged.
    # (1) Each model keeps a table: parameter set -> its fold MSEs (and its full-training-set refit). A set that
    #     another pruner's study, or the same study, already scored is not trained again: the study receives the
    #     same fold values, so the pruners decide the same way and Optuna suggests the same next trials.
    #     On pile, 66% of completed trials repeated an already-scored parameter set.
    # (2) Models are tuned 2 at a time in separate processes (Kaggle CPU: 2 cores x 2 threads), longest first.
    #     Each model still runs its 8 pruner studies in order, each with its own seed-42 sampler. Logs and all
    #     outputs (Excel sheets, plots, best_params, saved models) are gathered in the original model order.
    # Was: 80 studies one after another on one core, re-training every repeated parameter set.
    # =========================================================
    import io, contextlib, logging, shutil
    N_TUNE_WORKERS = 2
    _LONGEST_FIRST = ['PGBM', 'NGBoost', 'TabNet']
    refit_cache_dir = os.path.join(model_save_dir, "_refit_cache")

    def _params_key(params):
        return repr(sorted(params.items(), key=lambda kv: kv[0]))

    def _cv_mse(trial, model_name, model_class, params, pruner_name, rmse_trial_history, fold_cache):
        cached = fold_cache.setdefault(_params_key(params), [])  # FIX B14: fold MSEs already computed for these params
        fold_mse = []
        for k, (tr_idx, va_idx) in enumerate(cv_splits):
            if k < len(cached):
                fold_mse.append(cached[k])
            else:
                m = _fit_model(model_name, model_class, params, X_train[tr_idx], y_train[tr_idx])
                fold_mse.append(mean_squared_error(y_train[va_idx], _predict(model_name, m, X_train[va_idx])))
                cached.append(fold_mse[-1])
            # FIX 2: report progress after every fold so the pruner can stop bad trials early.
            # WilcoxonPruner compares per-fold values; the others use the running mean.
            trial.report(fold_mse[-1] if pruner_name == 'WilcoxonPruner' else float(np.mean(fold_mse)), k)
            if trial.should_prune():
                rmse_trial_history[pruner_name].append(np.nan)
                raise optuna.TrialPruned()
        return float(np.mean(fold_mse))

    def _keep_if_best(model_name, model_class, params, pruner_name, rmse, best_rmse_tracker, refit_cache):
        # ===== SAVE BEST MODEL (best CV score), refit on the FULL training set =====
        if rmse < best_rmse_tracker[(model_name, pruner_name)]:
            best_rmse_tracker[(model_name, pruner_name)] = rmse
            save_path = os.path.join(model_save_dir, f"{model_name}_{pruner_name}_BEST.pkl")
            _ensure_parent_dir(save_path)
            key = _params_key(params)
            if key not in refit_cache:  # FIX B14: refit each parameter set once, then copy
                refit_cache[key] = os.path.join(refit_cache_dir, f"{model_name}_{len(refit_cache)}.pkl")
                joblib.dump(_fit_model(model_name, model_class, params, X_train, y_train), _ensure_parent_dir(refit_cache[key]))
            shutil.copyfile(refit_cache[key], save_path)

    def _tune_one_model(model_name, model_class, param_space):
        # FIX B14: one model, its 8 pruner studies; returns its outputs + its printed log
        import __main__  # in a worker process: make the notebook's PGBM loss functions findable by name,
        __main__.mseloss_objective = mseloss_objective  # so saved PGBM models can be pickled (and are loaded
        __main__.rmseloss_metric = rmseloss_metric      # back in the notebook exactly as before)
        best_scores_m, pred_cols, timing_m, pruning_m = {}, {}, [], []
        best_rmse_tracker, fold_cache, refit_cache = {}, {}, {}
        _buf = io.StringIO()
        _handler = logging.StreamHandler(_buf)
        _handler.setFormatter(logging.Formatter("[%(levelname)1.1s %(asctime)s] %(message)s"))
        optuna.logging.disable_default_handler()
        optuna.logging.get_logger("optuna").addHandler(_handler)
        try:
            with contextlib.redirect_stdout(_buf), contextlib.redirect_stderr(_buf):

                rmse_trial_history = {p.__class__.__name__: [] for p in pruners}

                for pruner in pruners:
                    pruner_name = pruner.__class__.__name__
                    print(f"Running Optuna for {model_name} with {pruner_name}...")
                    start_time = time.time()

                    best_rmse_tracker[(model_name, pruner_name)] = np.inf

                    sampler = optunahub.load_module("samplers/auto_sampler").AutoSampler(seed=42)  # FIX 2: seeded -> reproducible
                    study = optuna.create_study(direction='minimize', sampler=sampler, pruner=pruner)

                    if model_name == 'PGBM':

                        def pgbm_objective(trial):
                            params = {
                                    'n_estimators': trial.suggest_categorical('n_estimators', [100, 200, 300, 500]),
                                    'learning_rate': trial.suggest_categorical('learning_rate', [0.01, 0.05, 0.1, 0.15]),
                                    'max_leaves': trial.suggest_int('max_leaves', 15, 63),
                                    'min_split_gain': trial.suggest_categorical('min_split_gain', [0.0, 0.1, 0.5, 1.0]),
                                    'reg_lambda': trial.suggest_categorical('reg_lambda', [0.1, 1.0, 5.0, 10.0]),
                                    'feature_fraction': trial.suggest_categorical('feature_fraction', [0.5, 0.7, 0.9, 1.0]),
                                    'bagging_fraction': trial.suggest_categorical('bagging_fraction', [0.5, 0.7, 0.9, 1.0]),
                                    'tree_correlation': trial.suggest_categorical('tree_correlation', [0.0, 0.1, 0.2, 0.3]),
                                    'min_data_in_leaf': trial.suggest_categorical('min_data_in_leaf', [3, 5, 10, 20]),
                                    'max_bin': trial.suggest_categorical('max_bin', [64, 128, 256]),
                                    'distribution': trial.suggest_categorical('distribution', ['normal', 'studentt', 'laplace']),
                                    'objective': 'mse',
                                    'metric': 'rmse',
                                    'random_state': 42,
                                    'verbose': 0
                                }

                            # FIX 1: CV on training data (was: fit on train, score on TEST)
                            mse = _cv_mse(trial, model_name, None, params, pruner_name, rmse_trial_history, fold_cache)
                            rmse = np.sqrt(mse)

                            rmse_trial_history[pruner_name].append(rmse)
                            _keep_if_best(model_name, None, params, pruner_name, rmse, best_rmse_tracker, refit_cache)

                            return mse

                        study.optimize(pgbm_objective, n_trials=50)

                    else:

                        def objective(trial):
                            params = {}
                            for key, values in param_space.items():
                                params[key] = trial.suggest_categorical(key, values)

                            # FIX 1: CV on training data (was: fit on train, score on TEST)
                            mse = _cv_mse(trial, model_name, model_class, params, pruner_name, rmse_trial_history, fold_cache)
                            rmse = np.sqrt(mse)

                            rmse_trial_history[pruner_name].append(rmse)
                            _keep_if_best(model_name, model_class, params, pruner_name, rmse, best_rmse_tracker, refit_cache)

                            return mse

                        study.optimize(objective, n_trials=50)

                    elapsed_time = time.time() - start_time

                    # FIX B13 (non-math): if the pruner stopped EVERY trial, no model was saved and the load below
                    # crashed (FileNotFoundError). Record the pair as "all trials pruned" and skip it; each model
                    # still has NopPruner (never prunes), so the other notebooks always find params for it.
                    if not any(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials):
                        print(f"  {model_name} / {pruner_name}: all {len(study.trials)} trials pruned - pair skipped")
                        pruning_m.append({
                            'Model': model_name, 'Pruner': pruner_name,
                            'Trials': len(study.trials), 'Completed': 0, 'Pruned': len(study.trials),
                            'Pruned_%': 100.0, 'Best_CV_RMSE': np.nan, 'Test_RMSE': np.nan,
                            'Tuning_Time_Seconds': round(elapsed_time, 1),
                        })
                        timing_m.append({'Model': model_name, 'Pruner': pruner_name, 'Tuning_Time_Seconds': elapsed_time})
                        continue

                    # Load frozen model (NO RETRAIN)
                    best_model = joblib.load(
                        os.path.join(model_save_dir, f"{model_name}_{pruner_name}_BEST.pkl")
                    )

                    y_pred = _predict(model_name, best_model, X_test)  # test set used ONCE, here

                    mse = mean_squared_error(y_test, y_pred)
                    rmse = np.sqrt(mse)
                    corr_coef = np.corrcoef(y_test, y_pred)[0, 1]


                    pred_cols[f'{model_name}_{pruner_name}_Predicted'] = y_pred

                    # FIX 2: pruning statistics for the new Excel sheet
                    n_complete = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
                    n_pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials)
                    pruning_m.append({
                        'Model': model_name, 'Pruner': pruner_name,
                        'Trials': len(study.trials), 'Completed': n_complete, 'Pruned': n_pruned,
                        'Pruned_%': round(100.0 * n_pruned / max(len(study.trials), 1), 1),
                        'Best_CV_RMSE': float(np.sqrt(study.best_value)),
                        'Test_RMSE': float(rmse),
                        'Tuning_Time_Seconds': round(elapsed_time, 1),
                    })

                    best_scores_m[(model_name, pruner_name)] = {
                        'best_score': study.best_value,  # FIX 1: CV MSE on train (was TEST MSE)
                        'cv_rmse': float(np.sqrt(study.best_value)),
                        'best_params': study.best_params,
                        'test_mse': mse,
                        'test_rmse': rmse,
                        'test_corr_coef': corr_coef,
                        'pruner': pruner_name
                    }

                    timing_m.append({
                        'Model': model_name,
                        'Pruner': pruner_name,
                        'Tuning_Time_Seconds': elapsed_time
                    })
        finally:
            optuna.logging.get_logger("optuna").removeHandler(_handler)
            optuna.logging.enable_default_handler()
        return {'log': _buf.getvalue(), 'rmse_trial_history': rmse_trial_history, 'best_scores': best_scores_m,
                'pred_cols': pred_cols, 'timing': timing_m, 'pruning': pruning_m}

    _order = sorted(models, key=lambda m: _LONGEST_FIRST.index(m) if m in _LONGEST_FIRST else len(_LONGEST_FIRST))
    optunahub.load_module("samplers/auto_sampler")  # download once, before the workers start
    if N_TUNE_WORKERS > 1:
        from joblib import Parallel, delayed
        try:
            from joblib import parallel_config
            _ctx = parallel_config(backend="loky", inner_max_num_threads=max(1, (os.cpu_count() or 2) // N_TUNE_WORKERS))
        except ImportError:
            _ctx = contextlib.nullcontext()
        print(f"Tuning {len(_order)} models, {N_TUNE_WORKERS} at a time (logs are printed per model when it finishes)...")
        with _ctx:
            _outs = Parallel(n_jobs=N_TUNE_WORKERS, backend="loky", batch_size=1)(
                delayed(_tune_one_model)(m, *models[m]) for m in _order)
        _results = dict(zip(_order, _outs))
    else:
        _results = {m: _tune_one_model(m, *models[m]) for m in _order}

    for model_name in models:  # FIX B14: gathered in the ORIGINAL model order
        _r = _results[model_name]
        print(_r['log'], end='')
        best_scores.update(_r['best_scores'])
        for _col, _vals in _r['pred_cols'].items():
            predictions_df[_col] = _vals
        timing_records.extend(_r['timing'])
        pruning_records.extend(_r['pruning'])
        rmse_trial_history = _r['rmse_trial_history']

        # RMSE plots & Excel writing (UNCHANGED)
        rmse_df = pd.DataFrame(rmse_trial_history)
        rmse_df.insert(0, "Trial", np.arange(1, len(rmse_df) + 1))

        if not os.path.exists(excel_path):
            pd.DataFrame().to_excel(excel_path)
        with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(excel_path)), engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            writer
            rmse_df.to_excel(writer, sheet_name=f"RMSE_Trials_{model_name}", index=False)
        # ================= SAVE RMSE PLOT =================
        plot_dir = os.path.dirname(excel_path)
        plot_path = os.path.join(plot_dir, f"RMSE_Trials_{model_name}.png")

        plt.figure(figsize=(10, 6))
        for pruner_name, values in rmse_trial_history.items():
            if len(values) > 0:   # <-- important safety check
                plt.plot(values, label=pruner_name, linewidth=2)

        plt.title(f"RMSE Variation Over Trials\n{model_name}")
        plt.xlabel("Trial Number")
        plt.ylabel("RMSE")
        plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
        plt.tight_layout()

        _ensure_parent_dir(plot_path)
        plt.savefig(_ensure_parent_dir(plot_path), dpi=100, bbox_inches="tight")
        plt.close()

        # ================= INSERT PLOT INTO EXCEL =================
        wb = load_workbook(_ensure_excel_file(excel_path))
        ws = wb[f"RMSE_Trials_{model_name}"]

        img = Image(plot_path)
        img.anchor = "J2"
        ws.add_image(img)

        _ensure_parent_dir(excel_path)
        wb.save(_ensure_parent_dir(excel_path))

    timing_df = pd.DataFrame(timing_records)

    if not os.path.exists(excel_path):
        pd.DataFrame().to_excel(excel_path)
    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(excel_path)), engine='openpyxl', mode='a') as writer:
        writer
        predictions_df.to_excel(writer, sheet_name='Predictions', index=False)
        writer
        timing_df.to_excel(writer, sheet_name='Tuning_Time', index=False)
        pd.DataFrame(pruning_records).to_excel(writer, sheet_name='Pruning_Summary', index=False)  # FIX 2

    return best_scores


best_scores_autosampler = hyperparameter_tuning_all(X_train, y_train, X_test, y_test, "./drive/MyDrive/Regression/noise_uncertainty_analysis/hyperparameter_tuning/test.xlsx")


Tuning 10 models, 2 at a time (logs are printed per model when it finishes)...


ninja: no work to do.


ninja: no work to do.


Running Optuna for Random Forest with MedianPruner...
[I 2026-11-22 13:39:49,981] A new study created in memory with name: no-name-2ffc2b04-6905-48e3-bc01-b571e3fc06fa
[I 2026-11-22 13:39:53,453] Trial 0 finished with value: 21.800097778543932 and parameters: {'n_estimators': 200, 'criterion': 'squared_error', 'max_depth': 20, 'min_samples_split': 0.01, 'min_samples_leaf': 3, 'min_weight_fraction_leaf': 0.0, 'max_features': 1.0, 'max_leaf_nodes': 200, 'min_impurity_decrease': 0.1, 'n_jobs': -1, 'random_state': 42, 'verbose': 0, 'warm_start': False, 'ccp_alpha': 0.1}. Best is trial 0 with value: 21.800097778543932.
/usr/local/lib/python3.13/dist-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the fu

Running Optuna for Gradient Boosting with MedianPruner...
[I 2026-11-22 13:52:42,838] A new study created in memory with name: no-name-da0e294b-6113-4f26-bfad-2759d7de73b3
[I 2026-11-22 13:52:44,790] Trial 0 finished with value: 24.858101173455122 and parameters: {'loss': 'absolute_error', 'learning_rate': 0.05, 'n_estimators': 700, 'subsample': 1.0, 'criterion': 'squared_error', 'min_samples_split': 5, 'min_samples_leaf': 1, 'min_weight_fraction_leaf': 0.0, 'max_depth': 5, 'min_impurity_decrease': 0.0, 'init': None, 'random_state': 42, 'max_features': None, 'alpha': 0.1, 'verbose': 0, 'max_leaf_nodes': 10, 'warm_start': False, 'validation_fraction': 0.1, 'n_iter_no_change': 10, 'tol': 0.001, 'ccp_alpha': 0.001}. Best is trial 0 with value: 24.858101173455122.
/usr/local/lib/python3.13/dist-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/usr/local/lib/python3.13/dis

Running Optuna for XGBoost with MedianPruner...
[I 2026-11-22 13:56:44,539] A new study created in memory with name: no-name-63721666-0fd7-415b-a8a9-6fe7ffa03493
[I 2026-11-22 13:56:45,055] Trial 0 finished with value: 46.55930578745672 and parameters: {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 7, 'min_child_weight': 1, 'gamma': 0.5, 'subsample': 0.5, 'colsample_bytree': 0.9, 'colsample_bylevel': 0.9, 'reg_alpha': 0.01, 'reg_lambda': 10, 'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1}. Best is trial 0 with value: 46.55930578745672.
/usr/local/lib/python3.13/dist-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-11-22 13:56:45,566]

... (3 outputs left out)

Running Optuna for NGBoost with MedianPruner...
[I 2026-11-22 13:03:11,833] A new study created in memory with name: no-name-b05c71ea-dc14-47db-8d22-5033bba04507
/usr/local/lib/python3.13/dist-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'ngboost.distns.normal.Normal'> which is of type type.
  warnings.warn(message)
/usr/local/lib/python3.13/dist-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'ngboost.scores.LogScore'> which is of type type.
  warnings.warn(message)
[I 2026-11-22 13:03:39,051] Trial 0 finished with value: 189.43999012304388 and parameters: {'n_estimators': 500, 'learning_rate': 0.03, 'natural_gradient': False, 'minibatch_frac': 0.9, 'col_sample': 1.0, 'Dist': <class 'ngboost.distns.normal.Normal'

Running Optuna for TabNet with MedianPruner...
[I 2026-11-22 13:26:35,293] A new study created in memory with name: no-name-9c4b2f12-9cf7-436a-87a8-38977fcfe811
/usr/local/lib/python3.13/dist-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains {'lr': 0.02} which is of type dict.
  warnings.warn(message)
/usr/local/lib/python3.13/dist-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains {'step_size': 10, 'gamma': 0.9} which is of type dict.
  warnings.warn(message)
/usr/local/lib/python3.13/dist-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.optim.lr_scheduler.StepLR'> which is of type type.
  warn

Running Optuna for HistGradientBoosting with MedianPruner...
[I 2026-11-22 14:19:12,019] A new study created in memory with name: no-name-d96e81d0-1c57-4567-a1d7-ec1c3288bbfb
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_a

Running Optuna for PGBM with MedianPruner...
[I 2026-11-22 13:03:11,720] A new study created in memory with name: no-name-bcb46bfa-ad58-4d41-84e9-4a4751a1754c
Training on CPU
Training on CPU
Training on CPU
Training on CPU
Training on CPU
Training on CPU
[I 2026-11-22 13:03:16,844] Trial 0 finished with value: 48.5265116066452 and parameters: {'n_estimators': 200, 'learning_rate': 0.01, 'max_leaves': 23, 'min_split_gain': 0.5, 'reg_lambda': 10.0, 'feature_fraction': 0.5, 'bagging_fraction': 0.5, 'tree_correlation': 0.1, 'min_data_in_leaf': 3, 'max_bin': 128, 'distribution': 'studentt'}. Best is trial 0 with value: 48.5265116066452.
/usr/local/lib/python3.13/dist-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface ca

In [13]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 19.291167396300658,
  'cv_rmse': 4.3921711483389005,
  'best_params': {'n_estimators': 300,
   'criterion': 'friedman_mse',
   'max_depth': None,
   'min_samples_split': 0.01,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.3,
   'max_leaf_nodes': 50,
   'min_impurity_decrease': 0.0,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.0},
  'test_mse': 17.22207284999383,
  'test_rmse': np.float64(4.149948535824731),
  'test_corr_coef': np.float64(0.9650589112564479),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 18.114017364046852,
  'cv_rmse': 4.256056550851604,
  'best_params': {'n_estimators': 300,
   'criterion': 'friedman_mse',
   'max_depth': 20,
   'min_samples_split': 2,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.3,
   'max_leaf_nodes': 100,
   'min_impurity_decrease': 0.01,
  

In [14]:
# FIX K2: save the tuning results so the other notebooks LOAD them instead of copy-pasting.
# best_params.pkl  -> loaded by the other notebooks (keeps Python objects such as NGBoost's Normal)
# best_params.json -> the same content in readable form, for checking by eye
import pickle, json
_out_dir = "./drive/MyDrive/Regression/noise_uncertainty_analysis/hyperparameter_tuning"
os.makedirs(_out_dir, exist_ok=True)
with open(os.path.join(_out_dir, "best_params.pkl"), "wb") as f:
    pickle.dump(best_scores_autosampler, f)
with open(os.path.join(_out_dir, "best_params.json"), "w") as f:
    json.dump({f"{m} | {p}": v for (m, p), v in best_scores_autosampler.items()}, f, indent=1, default=str)
print("Saved tuning results to", _out_dir)


Saved tuning results to ./drive/MyDrive/Regression/noise_uncertainty_analysis/hyperparameter_tuning


# **Best Model Analysis**

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def get_best_models_and_predict(best_scores_autosampler, X_train, y_train, X_test, y_test, file_path):
    # Convert input data to NumPy arrays
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    y_test = np.array(y_test)

    # Mapping for model creation based on dictionary keys
    model_mapping = {
        'Random Forest': RandomForestRegressor,
        'Gradient Boosting': GradientBoostingRegressor,
        'XGBoost': XGBRegressor,
        'LightGBM': LGBMRegressor,
        'CatBoost': CatBoostRegressor,
        'GPBoost': GPBoostRegressor,
        'NGBoost': NGBRegressor,
        'TabNet': TabNetRegressor,
        'HistGradientBoosting': HistGradientBoostingRegressor,
        'PGBM': PGBM  # PGBM is handled separately
    }

    # Dictionary to store the best model for each type
    best_models = {}

    # Iterate over the dictionary to find the best pruner for each model type
    for (model_name, pruner), params in best_scores_autosampler.items():
        current_score = params.get('best_score', np.inf)  # FIX 1: pick by CV score (was test_mse)
        if model_name not in best_models or current_score < best_models[model_name]['score']:
            best_models[model_name] = {
                'score': current_score,
                'params': params['best_params'],
                'pruner': pruner
            }

    # Prepare a DataFrame to store predictions
    df = pd.read_csv(file_path)

    # Iterate over the best models to train and predict
    for model_name, model_info in best_models.items():
        best_params = model_info['params']
        model_class = model_mapping.get(model_name)

        if model_class is None:
            print(f"Model {model_name} is not supported or not available.")
            continue

        # Handle specific parameters or settings for model if needed
        if model_name == 'CatBoost':
            best_params.pop('verbose', None)  # Remove 'verbose' for CatBoost

        # Create an instance of the best model with the best parameters
        if model_name == 'PGBM':
            model = model_class()
            model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)
            predictions = model.predict(X_test)
        elif model_name == 'TabNet':
            model = model_class(**best_params)
            model.fit(X_train, y_train.reshape(-1, 1))
            predictions = model.predict(X_test)
            predictions = predictions.ravel()
        else:
            model = model_class(**best_params)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)

        # Add predictions to the DataFrame
        df[f'{model_name} Predictions'] = predictions

        # Plot actual vs. predicted
        plt.figure(figsize=(10, 6))
        plt.scatter(y_test, predictions, alpha=0.6)
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', color='red', lw=2)
        plt.xlabel(f"Actual {TARGET_LABEL}")
        plt.ylabel(f"Predicted {TARGET_LABEL}")
        plt.title(f"Actual vs. Predicted Values ({model_name})")
        plt.grid(True)
        plt.tight_layout()

        # Save the plot temporarily
        plot_path = f'temp_plot_{model_name}.png'
        _ensure_parent_dir(plot_path)
        plt.savefig(_ensure_parent_dir(plot_path))
        plt.close()

    # ✅ NEW OUTPUT DIRECTORY
    output_dir = "./drive/MyDrive/Regression/noise_uncertainty_analysis/hyperparameter_tuning/"
    os.makedirs(output_dir, exist_ok=True)

    output_excel_path = os.path.join(
        output_dir,
        os.path.basename(file_path).replace('.csv', '_results.xlsx')
    )

    # Save predictions and plots to Excel
    with pd.ExcelWriter(_ensure_parent_dir(output_excel_path), engine='xlsxwriter') as writer:
        # Write data to Excel
        writer
        df.to_excel(writer, sheet_name='data', index=False)

        # Get the xlsxwriter objects
        workbook = writer.book

        # Insert each plot into a separate worksheet
        for model_name in best_models.keys():
            short_model_name = ''.join([word[0] for word in model_name.split()])
            sheet_name = f'{short_model_name}_Plot'

            worksheet = workbook.add_worksheet(sheet_name)
            writer.sheets[sheet_name] = worksheet
            plot_path = f'temp_plot_{model_name}.png'
            worksheet.insert_image('A1', plot_path)

    # Clean up temporary plot files
    for model_name in best_models.keys():
        if os.path.exists(str(f'temp_plot_{model_name}.png')): os.remove(str(f'temp_plot_{model_name}.png'))

    return df, best_models

# Call the function
df, best_models = get_best_models_and_predict(best_scores_autosampler, X_train, y_train, X_test, y_test, "./drive/MyDrive/Regression/noise_uncertainty_analysis/data/test.csv")

0:	learn: 13.3481525	total: 47.9ms	remaining: 9.53s
1:	learn: 12.5680984	total: 48.6ms	remaining: 4.81s
2:	learn: 11.7764825	total: 48.9ms	remaining: 3.21s
3:	learn: 11.1760343	total: 49.1ms	remaining: 2.41s
4:	learn: 10.4602488	total: 49.4ms	remaining: 1.93s
5:	learn: 9.8788619	total: 49.6ms	remaining: 1.6s
6:	learn: 9.3371825	total: 49.9ms	remaining: 1.38s
7:	learn: 8.7939343	total: 50.2ms	remaining: 1.2s
8:	learn: 8.3318924	total: 50.5ms	remaining: 1.07s
9:	learn: 7.9561784	total: 50.7ms	remaining: 963ms
10:	learn: 7.5787826	total: 50.9ms	remaining: 875ms
11:	learn: 7.2250490	total: 51.2ms	remaining: 802ms
12:	learn: 6.8769568	total: 51.4ms	remaining: 740ms
13:	learn: 6.6162414	total: 51.7ms	remaining: 687ms
14:	learn: 6.3013501	total: 51.9ms	remaining: 641ms
15:	learn: 6.0635657	total: 52.2ms	remaining: 600ms
16:	learn: 5.8170395	total: 52.4ms	remaining: 564ms
17:	learn: 5.6119629	total: 52.7ms	remaining: 533ms
18:	learn: 5.3999705	total: 52.9ms	remaining: 504ms
19:	learn: 5.215568

Training on CPU
Estimator 0/100, Train metric: 13.5823


Estimator 1/100, Train metric: 12.9859
Estimator 2/100, Train metric: 12.4356
Estimator 3/100, Train metric: 11.9029
Estimator 4/100, Train metric: 11.4019
Estimator 5/100, Train metric: 10.9165
Estimator 6/100, Train metric: 10.4506
Estimator 7/100, Train metric: 10.0087
Estimator 8/100, Train metric: 9.5896
Estimator 9/100, Train metric: 9.1922
Estimator 10/100, Train metric: 8.8209
Estimator 11/100, Train metric: 8.4728
Estimator 12/100, Train metric: 8.1313
Estimator 13/100, Train metric: 7.8142
Estimator 14/100, Train metric: 7.5028
Estimator 15/100, Train metric: 7.2011
Estimator 16/100, Train metric: 6.9205
Estimator 17/100, Train metric: 6.6474


Estimator 18/100, Train metric: 6.3886
Estimator 19/100, Train metric: 6.1544
Estimator 20/100, Train metric: 5.9238
Estimator 21/100, Train metric: 5.7004
Estimator 22/100, Train metric: 5.5008
Estimator 23/100, Train metric: 5.2968
Estimator 24/100, Train metric: 5.1073
Estimator 25/100, Train metric: 4.9373
Estimator 26/100, Train metric: 4.7665
Estimator 27/100, Train metric: 4.6124
Estimator 28/100, Train metric: 4.4596
Estimator 29/100, Train metric: 4.3087
Estimator 30/100, Train metric: 4.1734
Estimator 31/100, Train metric: 4.0334
Estimator 32/100, Train metric: 3.9122


Estimator 33/100, Train metric: 3.7774
Estimator 34/100, Train metric: 3.6682
Estimator 35/100, Train metric: 3.5549
Estimator 36/100, Train metric: 3.4547
Estimator 37/100, Train metric: 3.3491
Estimator 38/100, Train metric: 3.2461
Estimator 39/100, Train metric: 3.1490
Estimator 40/100, Train metric: 3.0611
Estimator 41/100, Train metric: 2.9699
Estimator 42/100, Train metric: 2.8949
Estimator 43/100, Train metric: 2.8201


Estimator 44/100, Train metric: 2.7370
Estimator 45/100, Train metric: 2.6646
Estimator 46/100, Train metric: 2.6019
Estimator 47/100, Train metric: 2.5310
Estimator 48/100, Train metric: 2.4609
Estimator 49/100, Train metric: 2.3927
Estimator 50/100, Train metric: 2.3280
Estimator 51/100, Train metric: 2.2702
Estimator 52/100, Train metric: 2.2238
Estimator 53/100, Train metric: 2.1783


... (2 outputs left out)

Estimator 70/100, Train metric: 1.5903
Estimator 71/100, Train metric: 1.5584
Estimator 72/100, Train metric: 1.5285
Estimator 73/100, Train metric: 1.5093
Estimator 74/100, Train metric: 1.4875
Estimator 75/100, Train metric: 1.4608
Estimator 76/100, Train metric: 1.4425


Estimator 77/100, Train metric: 1.4257
Estimator 78/100, Train metric: 1.4030
Estimator 79/100, Train metric: 1.3868
Estimator 80/100, Train metric: 1.3716
Estimator 81/100, Train metric: 1.3514
Estimator 82/100, Train metric: 1.3301
Estimator 83/100, Train metric: 1.3184
Estimator 84/100, Train metric: 1.3006


Estimator 85/100, Train metric: 1.2895
Estimator 86/100, Train metric: 1.2736
Estimator 87/100, Train metric: 1.2586
Estimator 88/100, Train metric: 1.2463
Estimator 89/100, Train metric: 1.2324
Estimator 90/100, Train metric: 1.2171
Estimator 91/100, Train metric: 1.1993
Estimator 92/100, Train metric: 1.1860


Estimator 93/100, Train metric: 1.1720
Estimator 94/100, Train metric: 1.1546
Estimator 95/100, Train metric: 1.1393
Estimator 96/100, Train metric: 1.1250
Estimator 97/100, Train metric: 1.1154
Estimator 98/100, Train metric: 1.0994
Estimator 99/100, Train metric: 1.0854


In [16]:
plot_best_scores(best_scores_autosampler,"./drive/MyDrive/Regression/noise_uncertainty_analysis/hyperparameter_tuning/test_results.xlsx")

In [17]:
# FIX B10 (non-math): `model_classes` was used by generate_interpretml_explanations_summary_pruners but never
# defined in this notebook (NameError in the last cell, also in v1.2). Same mapping as in the other notebooks.
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}
generate_interpretml_explanations_summary_pruners(best_scores_autosampler, X_train, y_train, x_test, feature_names,excel_file_path = "./drive/MyDrive/Regression/noise_uncertainty_analysis/hyperparameter_tuning/test_results.xlsx")

Using 168 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/60 [00:00<?, ?it/s]

Using 168 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/60 [00:00<?, ?it/s]

Using 168 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/60 [00:00<?, ?it/s]

... (23 outputs left out)

  0%|          | 0/60 [00:00<?, ?it/s]

Using 168 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/60 [00:00<?, ?it/s]

Model PGBM is not supported or not available.
